In [ ]:
############################################################
# 0. 라이브러리 불러오기
############################################################
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


############################################################
# 1. 파일 경로 설정
############################################################

TRAIN_PATH            = "/content/drive/MyDrive/train/train.csv"
TRAIN_META_WS_PATH    = "/content/drive/MyDrive/train/meta/TRAIN_전국도매_2018-2021.csv"
TRAIN_META_AC_PATH    = "/content/drive/MyDrive/train/meta/TRAIN_산지공판장_2018-2021.csv"
TEST_GLOB             = "/content/drive/MyDrive/test/TEST_*.csv"
META_WS_GLOB          = "/content/drive/MyDrive/test/meta/TEST_전국도매_*.csv"
META_AC_GLOB          = "/content/drive/MyDrive/test/meta/TEST_산지공판장_*.csv"
SUBMISSION_PATH       = "/content/drive/MyDrive/sample_submission.csv"
OUTPUT_PATH           = "meta_submission_v4.csv"


############################################################
# 2. 예측 대상 조합
############################################################
TARGET_MAP = {
    "건고추":       {"품종명": ["화건"],          "거래단위": "30 kg",     "등급": "상품"},
    "사과":         {"품종명": ["홍로", "후지"],   "거래단위": "10 개",     "등급": "상품"},
    "감자":         {"품종명": ["감자 수미"],      "거래단위": "20키로상자", "등급": "상"},
    "배":           {"품종명": ["신고"],           "거래단위": "10 개",     "등급": "상품"},
    "깐마늘(국산)": {"품종명": ["깐마늘(국산)"],   "거래단위": "20 kg",     "등급": "상품"},
    "무":           {"품종명": ["무"],             "거래단위": "20키로상자", "등급": "상"},
    "상추":         {"품종명": ["청"],             "거래단위": "100 g",     "등급": "상품"},
    "배추":         {"품종명": ["배추"],           "거래단위": "10키로망대", "등급": "상"},
    "양파":         {"품종명": ["양파"],           "거래단위": "1키로",      "등급": "상"},
    "대파":         {"품종명": ["대파(일반)"],     "거래단위": "1키로단",    "등급": "상"},
}

# train.csv 품목명 -> 메타데이터 품목명 매핑
# 건고추는 메타에 없으므로 None
META_ITEM_MAP = {
    "건고추":       None,
    "사과":         "사과",
    "감자":         "감자",
    "배":           "배",
    "깐마늘(국산)": "마늘",
    "무":           "무",
    "상추":         "상추",
    "배추":         "배추",
    "양파":         "양파",
    "대파":         "대파",
}


############################################################
# 3. 공통 유틸 함수
############################################################
def clean_columns(df):
    """컬럼명 앞뒤 공백 제거, 중복 공백 제거"""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def soon_to_date(x):
    """201801상순 -> 2018-01-01"""
    x     = str(x).strip()
    year  = int(x[:4])
    month = int(x[4:6])
    soon  = x[6:]
    if soon == "상순":    day = 1
    elif soon == "중순":  day = 11
    else:                 day = 21
    return pd.Timestamp(year=year, month=month, day=day)


def make_time_features(df):
    """시점 -> date/year/month/day/month_sin/month_cos"""
    df = df.copy()
    df["date"]      = df["시점"].apply(soon_to_date)
    df["year"]      = df["date"].dt.year
    df["month"]     = df["date"].dt.month
    df["day"]       = df["date"].dt.day
    df["soon"]      = df["시점"].str[-2:]
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def fill_normal_price(df):
    """평년 평균가격 0 -> 결측 처리 후 ffill -> bfill -> 중앙값 -> 1"""
    df = df.copy()
    if "평년 평균가격(원)" in df.columns:
        df["평년 평균가격(원)"] = df["평년 평균가격(원)"].replace(0, np.nan)
        df["평년 평균가격(원)"] = df["평년 평균가격(원)"].ffill().bfill()
        if df["평년 평균가격(원)"].isnull().all():
            df["평년 평균가격(원)"] = 1
        else:
            df["평년 평균가격(원)"] = df["평년 평균가격(원)"].fillna(
                df["평년 평균가격(원)"].median()
            )
    return df


def make_series_features(df):
    """lag / rolling / diff feature 생성 (date 기준 정렬 후)"""
    df = df.copy()
    df = df.sort_values("date").reset_index(drop=True)

    for lag in range(1, 9):
        df[f"price_lag_{lag}"] = df["평균가격(원)"].shift(lag)

    df["lag_3"] = df["평균가격(원)"].shift(3)
    df["lag_5"] = df["평균가격(원)"].shift(5)
    df["lag_7"] = df["평균가격(원)"].shift(7)

    df["rolling_mean_3"] = df["평균가격(원)"].shift(1).rolling(3).mean()
    df["rolling_mean_5"] = df["평균가격(원)"].shift(1).rolling(5).mean()
    df["rolling_std_5"]  = df["평균가격(원)"].shift(1).rolling(5).std()

    df["diff_1"] = df["평균가격(원)"].diff(1)
    df["diff_3"] = df["평균가격(원)"].diff(3)

    df["평년대비_차이"] = df["평균가격(원)"] - df["평년 평균가격(원)"]
    df["평년대비_비율"] = df["평균가격(원)"] / df["평년 평균가격(원)"]

    return df


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


META_COLS = [
    "meta_ws_총반입량", "meta_ws_평균가", "meta_ws_경매건수",
    "meta_ws_전순평균가", "meta_ws_전달평균가", "meta_ws_전년평균가", "meta_ws_평년평균가",
    "meta_ws_반입량_lag1", "meta_ws_반입량_rolling3", "meta_ws_가격_rolling3",
    "meta_ac_총반입량", "meta_ac_평균가", "meta_ac_경매건수",
    "meta_ac_전순평균가", "meta_ac_전달평균가", "meta_ac_전년평균가", "meta_ac_평년평균가",
    "meta_ac_반입량_lag1", "meta_ac_반입량_rolling3", "meta_ac_가격_rolling3",
]

def get_feature_cols():
    base = [
        "평균가격(원)", "평년 평균가격(원)", "평년대비_차이", "평년대비_비율",
        "year", "month", "day", "month_sin", "month_cos",
        "lag_3", "lag_5", "lag_7",
        "rolling_mean_3", "rolling_mean_5", "rolling_std_5",
        "diff_1", "diff_3",
    ]
    for lag in range(1, 9):
        base.append(f"price_lag_{lag}")
    return base + META_COLS


############################################################
# 4. 메타데이터 전처리 함수
############################################################
def clean_meta_wholesale(df):
    """
    전국도매 가격 구조 오류 처리
    - cond1 위반(고가>=중가>=저가): 위반 행의 고/중/저가 NaN
    - cond2 위반(최고가>=중간가>=최저가): 위반 행 NaN
    - cond3 위반(최고가>=평균가>=최저가): 평균가 NaN
    """
    df = df.copy()
    df = clean_columns(df)

    cond1_ok = (
        (df["고가(20%) 평균가"] >= df["중가(60%) 평균가"]) &
        (df["중가(60%) 평균가"] >= df["저가(20%) 평균가"])
    )
    df.loc[~cond1_ok, ["고가(20%) 평균가", "중가(60%) 평균가", "저가(20%) 평균가"]] = np.nan

    cond2_ok = (
        (df["최고가(원/kg)"] >= df["중간가(원/kg)"]) &
        (df["중간가(원/kg)"] >= df["최저가(원/kg)"])
    )
    df.loc[~cond2_ok, ["최고가(원/kg)", "중간가(원/kg)", "최저가(원/kg)"]] = np.nan

    cond3_ok = (
        (df["최고가(원/kg)"] >= df["평균가(원/kg)"]) &
        (df["평균가(원/kg)"] >= df["최저가(원/kg)"])
    )
    df.loc[~cond3_ok, "평균가(원/kg)"] = np.nan

    return df


def clean_meta_auction(df):
    """산지공판장 가격 구조 오류 처리 (cond2/cond3)"""
    df = df.copy()
    df = clean_columns(df)

    cond2_ok = (
        (df["최고가(원/kg)"] >= df["중간가(원/kg)"]) &
        (df["중간가(원/kg)"] >= df["최저가(원/kg)"])
    )
    df.loc[~cond2_ok, ["최고가(원/kg)", "중간가(원/kg)", "최저가(원/kg)"]] = np.nan

    cond3_ok = (
        (df["최고가(원/kg)"] >= df["평균가(원/kg)"]) &
        (df["평균가(원/kg)"] >= df["최저가(원/kg)"])
    )
    df.loc[~cond3_ok, "평균가(원/kg)"] = np.nan

    return df


def aggregate_meta(df, prefix):
    """
    품목별 시점 집계
    - 반입량 가중 평균가 사용
    - prefix: 'ws'(전국도매) 또는 'ac'(산지공판장)
    """
    df = df.copy()
    df["거래금액_proxy"] = df["평균가(원/kg)"].fillna(0) * df["총반입량(kg)"].fillna(0)

    agg = df.groupby(["시점", "품목명"]).agg(
        **{f"meta_{prefix}_총반입량":  ("총반입량(kg)", "sum")},
        **{f"meta_{prefix}_거래금액합": ("거래금액_proxy", "sum")},
        **{f"meta_{prefix}_경매건수":  ("경매 건수", "sum")},
        **{f"meta_{prefix}_전순평균가": ("전순 평균가격(원) PreVious SOON", "mean")},
        **{f"meta_{prefix}_전달평균가": ("전달 평균가격(원) PreVious MMonth", "mean")},
        **{f"meta_{prefix}_전년평균가": ("전년 평균가격(원) PreVious YeaR", "mean")},
        **{f"meta_{prefix}_평년평균가": ("평년 평균가격(원) Common Year SOON", "mean")},
    ).reset_index()

    vol_col = f"meta_{prefix}_총반입량"
    agg[f"meta_{prefix}_평균가"] = np.where(
        agg[vol_col] > 0,
        agg[f"meta_{prefix}_거래금액합"] / agg[vol_col],
        np.nan
    )
    agg = agg.drop(columns=[f"meta_{prefix}_거래금액합"])

    return agg


def add_meta_series_features(agg, prefix, date_col="date"):
    """
    집계된 메타에 lag/rolling feature 추가
    date_col 기준 정렬 후 생성
    """
    agg = agg.copy().sort_values(date_col).reset_index(drop=True)

    vol_col   = f"meta_{prefix}_총반입량"
    price_col = f"meta_{prefix}_평균가"

    agg[f"meta_{prefix}_반입량_lag1"]     = agg[vol_col].shift(1)
    agg[f"meta_{prefix}_반입량_rolling3"] = agg[vol_col].shift(1).rolling(3).mean()
    agg[f"meta_{prefix}_가격_rolling3"]   = agg[price_col].shift(1).rolling(3).mean()

    return agg


############################################################
# 5. TRAIN 메타데이터 로드 및 집계
############################################################
print("=" * 70)
print("[TRAIN 메타데이터 로드 중...]")

# 전국도매
ws_raw = pd.read_csv(TRAIN_META_WS_PATH)
ws_raw = clean_meta_wholesale(ws_raw)
train_ws_agg = aggregate_meta(ws_raw, "ws")

# 시점을 date로 변환하여 lag/rolling 생성
train_ws_agg["date"] = train_ws_agg["시점"].apply(soon_to_date)
train_ws_list = []
for item_name, g in train_ws_agg.groupby("품목명"):
    g = add_meta_series_features(g, "ws", date_col="date")
    train_ws_list.append(g)
train_ws_agg = pd.concat(train_ws_list).reset_index(drop=True)

# 산지공판장
ac_raw = pd.read_csv(TRAIN_META_AC_PATH)
ac_raw = clean_meta_auction(ac_raw)
train_ac_agg = aggregate_meta(ac_raw, "ac")

train_ac_agg["date"] = train_ac_agg["시점"].apply(soon_to_date)
train_ac_list = []
for item_name, g in train_ac_agg.groupby("품목명"):
    g = add_meta_series_features(g, "ac", date_col="date")
    train_ac_list.append(g)
train_ac_agg = pd.concat(train_ac_list).reset_index(drop=True)

print(f"  전국도매 집계 완료: {train_ws_agg.shape}")
print(f"  산지공판장 집계 완료: {train_ac_agg.shape}")


############################################################
# 6. TRAIN 데이터 로드
############################################################
train_raw  = pd.read_csv(TRAIN_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)
train_raw  = clean_columns(train_raw)
sample_sub = clean_columns(sample_sub)

print(f"\ntrain_raw shape: {train_raw.shape}")
print(f"sample_sub shape: {sample_sub.shape}")


############################################################
# 7. 품목별 학습 데이터 구성 (메타 merge 포함)
############################################################
def build_item_train_df(train_raw, item, config, train_ws_agg, train_ac_agg):
    varieties  = config["품종명"]
    unit       = config["거래단위"]
    grade      = config["등급"]
    meta_item  = META_ITEM_MAP.get(item)

    df = train_raw[
        (train_raw["품목명"] == item) &
        (train_raw["품종명"].isin(varieties)) &
        (train_raw["거래단위"] == unit) &
        (train_raw["등급"] == grade)
    ].copy()

    out_list = []
    for variety in varieties:
        temp = df[df["품종명"] == variety].copy()
        if len(temp) == 0:
            continue

        temp = make_time_features(temp)
        temp = fill_normal_price(temp)
        temp = make_series_features(temp)

        # -------------------------------------------------
        # 메타 feature merge (건고추 제외)
        # -------------------------------------------------
        if meta_item is not None:
            # 전국도매
            ws_item = train_ws_agg[train_ws_agg["품목명"] == meta_item].copy()
            ws_merge_cols = ["date"] + [c for c in ws_item.columns if c.startswith("meta_ws_")]
            temp = temp.merge(ws_item[ws_merge_cols], on="date", how="left")

            # 산지공판장
            ac_item = train_ac_agg[train_ac_agg["품목명"] == meta_item].copy()
            ac_merge_cols = ["date"] + [c for c in ac_item.columns if c.startswith("meta_ac_")]
            temp = temp.merge(ac_item[ac_merge_cols], on="date", how="left")

        # direct target
        temp["target_h1"] = temp["평균가격(원)"].shift(-1)
        temp["target_h2"] = temp["평균가격(원)"].shift(-2)
        temp["target_h3"] = temp["평균가격(원)"].shift(-3)

        out_list.append(temp)

    if len(out_list) == 0:
        return pd.DataFrame()

    return pd.concat(out_list, axis=0).reset_index(drop=True)


############################################################
# 8. 학습
############################################################
def train_validate_direct_models(item_df, feature_cols):
    item_df = item_df.copy()
    item_df = item_df.sort_values(["품종명", "date"]).reset_index(drop=True)

    results = {}
    models  = {}

    for horizon in [1, 2, 3]:
        target_col = f"target_h{horizon}"
        df_h = item_df.dropna(subset=[target_col]).copy()

        # 실제로 존재하는 feature만 사용
        use_cols = [c for c in feature_cols if c in df_h.columns]

        tr_parts, va_parts = [], []
        for _, g in df_h.groupby("품목명"):
            g = g.sort_values("date")
            if len(g) <= 3:
                continue
            tr_parts.append(g.iloc[:-3])
            va_parts.append(g.iloc[-3:])

        tr_df = pd.concat(tr_parts).reset_index(drop=True)
        va_df = pd.concat(va_parts).reset_index(drop=True)

        X_train = tr_df[use_cols].fillna(0)
        y_train = tr_df[target_col]
        X_valid = va_df[use_cols].fillna(0)
        y_valid = va_df[target_col]

        model = XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42
        )
        model.fit(X_train, y_train)
        pred_valid = model.predict(X_valid)

        results[horizon] = {
            "MAE":      mean_absolute_error(y_valid, pred_valid),
            "RMSE":     rmse(y_valid, pred_valid),
            "n_train":  len(tr_df),
            "n_valid":  len(va_df),
            "use_cols": use_cols,
        }

        # 전체 데이터로 재학습
        X_full = df_h[use_cols].fillna(0)
        y_full = df_h[target_col]
        model.fit(X_full, y_full)
        models[horizon] = model

    return models, results


feature_cols = get_feature_cols()
all_models   = {}
all_results  = {}

for item, config in TARGET_MAP.items():
    print("\n" + "=" * 70)
    print(f"[{item}] 학습 시작")

    item_df = build_item_train_df(
        train_raw, item, config, train_ws_agg, train_ac_agg
    )
    print(f"  데이터 shape: {item_df.shape}")

    # 메타 컬럼 포함 여부 확인
    meta_in = [c for c in item_df.columns if c.startswith("meta_")]
    print(f"  메타 feature 수: {len(meta_in)}")

    models, results = train_validate_direct_models(item_df, feature_cols)

    all_models[item]  = {"models": models, "results": results}
    all_results[item] = results

    for h in [1, 2, 3]:
        print(
            f"  h={h} | "
            f"train={results[h]['n_train']} | "
            f"valid={results[h]['n_valid']} | "
            f"MAE={results[h]['MAE']:.4f} | "
            f"RMSE={results[h]['RMSE']:.4f} | "
            f"features={len(results[h]['use_cols'])}"
        )


############################################################
# 9. validation 성능 요약
############################################################
print("\n" + "=" * 70)
print("[품목별 Validation Score 요약]")
summary_rows = []
for item, res in all_results.items():
    for h in [1, 2, 3]:
        summary_rows.append({
            "품목명": item, "horizon": h,
            "MAE": res[h]["MAE"], "RMSE": res[h]["RMSE"]
        })
print(pd.DataFrame(summary_rows).to_string(index=False))


############################################################
# 10. TEST 메타데이터 로드 함수
############################################################
def load_test_meta_wholesale(path):
    """test 전국도매 메타 로드 + 집계 + lag/rolling 생성"""
    df = pd.read_csv(path)
    df = clean_meta_wholesale(df)
    agg = aggregate_meta(df, "ws")

    # test 시점(T-8순~T순)을 t_idx로 변환 후 정렬
    def parse_t_idx(x):
        x = str(x).strip().replace("순", "")
        if x == "T":           return 0
        if x.startswith("T-"): return int(x.replace("T", ""))
        if x.startswith("T+"): return int(x.replace("T", ""))
        raise ValueError(f"Unknown: {x}")

    agg["t_idx"] = agg["시점"].apply(parse_t_idx)

    # 가상 date 부여 (lag/rolling 계산용)
    base = pd.Timestamp(2000, 1, 21)
    agg["date"] = agg["t_idx"].apply(lambda v: base + pd.Timedelta(days=10 * v))

    result_list = []
    for item_name, g in agg.groupby("품목명"):
        g = add_meta_series_features(g, "ws", date_col="date")
        result_list.append(g)

    return pd.concat(result_list).reset_index(drop=True)


def load_test_meta_auction(path):
    """test 산지공판장 메타 로드 + 집계 + lag/rolling 생성"""
    df = pd.read_csv(path)
    df = clean_meta_auction(df)
    agg = aggregate_meta(df, "ac")

    def parse_t_idx(x):
        x = str(x).strip().replace("순", "")
        if x == "T":           return 0
        if x.startswith("T-"): return int(x.replace("T", ""))
        if x.startswith("T+"): return int(x.replace("T", ""))
        raise ValueError(f"Unknown: {x}")

    agg["t_idx"] = agg["시점"].apply(parse_t_idx)
    base = pd.Timestamp(2000, 1, 21)
    agg["date"] = agg["t_idx"].apply(lambda v: base + pd.Timedelta(days=10 * v))

    result_list = []
    for item_name, g in agg.groupby("품목명"):
        g = add_meta_series_features(g, "ac", date_col="date")
        result_list.append(g)

    return pd.concat(result_list).reset_index(drop=True)


############################################################
# 11. test 예측
############################################################
test_files = sorted(glob.glob(TEST_GLOB))
print(f"\n[test 파일 개수] {len(test_files)}")

submission_rows = []

for test_path in test_files:
    base_name = os.path.basename(test_path).replace(".csv", "")
    file_num  = base_name.replace("TEST_", "")
    print(f"predicting: {base_name}")

    ws_path = f"/content/drive/MyDrive/test/meta/TEST_전국도매_{file_num}.csv"
    ac_path = f"/content/drive/MyDrive/test/meta/TEST_산지공판장_{file_num}.csv"

    ws_meta = load_test_meta_wholesale(ws_path) if os.path.exists(ws_path) else None
    ac_meta = load_test_meta_auction(ac_path)   if os.path.exists(ac_path) else None

    horizon_preds = {1: {}, 2: {}, 3: {}}

    for item, config in TARGET_MAP.items():
        # -----------------------------------------------
        # 기본 test row 구성
        # -----------------------------------------------
        df_test   = pd.read_csv(test_path)
        df_test   = clean_columns(df_test)
        varieties = config["품종명"]
        unit      = config["거래단위"]
        grade     = config["등급"]

        temp = df_test[
            (df_test["품목명"] == item) &
            (df_test["품종명"].isin(varieties)) &
            (df_test["거래단위"] == unit) &
            (df_test["등급"] == grade)
        ].copy()

        out_list = []
        for variety in varieties:
            g = temp[temp["품종명"] == variety].copy()
            if len(g) == 0:
                continue

            def parse_t_idx(x):
                x = str(x).strip().replace("순", "")
                if x == "T":           return 0
                if x.startswith("T-"): return int(x.replace("T", ""))
                if x.startswith("T+"): return int(x.replace("T", ""))
                raise ValueError(f"Unknown: {x}")

            g["t_idx"] = g["시점"].apply(parse_t_idx)
            g = g.sort_values("t_idx").reset_index(drop=True)

            base_date  = pd.Timestamp(2000, 1, 21)
            g["date"]  = g["t_idx"].apply(lambda v: base_date + pd.Timedelta(days=10 * v))
            g["year"]  = g["date"].dt.year
            g["month"] = g["date"].dt.month
            g["day"]   = g["date"].dt.day
            g["month_sin"] = np.sin(2 * np.pi * g["month"] / 12)
            g["month_cos"] = np.cos(2 * np.pi * g["month"] / 12)

            g = fill_normal_price(g)
            g = make_series_features(g)
            out_list.append(g)

        if len(out_list) == 0:
            for h in [1, 2, 3]:
                horizon_preds[h][item] = np.nan
            continue

        last_rows  = pd.concat(out_list).reset_index(drop=True)
        meta_item  = META_ITEM_MAP.get(item)

        # -----------------------------------------------
        # 메타 feature merge (T순 row 기준)
        # -----------------------------------------------
        if meta_item is not None:
            if ws_meta is not None:
                ws_item = ws_meta[ws_meta["품목명"] == meta_item].copy()
                ws_T    = ws_item[ws_item["t_idx"] == 0]
                ws_feat_cols = [c for c in ws_T.columns if c.startswith("meta_ws_")]
                if len(ws_T) > 0:
                    for col in ws_feat_cols:
                        last_rows[col] = ws_T[col].iloc[0]

            if ac_meta is not None:
                # Corrected line: Filter ac_meta using ac_meta["품목명"]
                ac_item = ac_meta[ac_meta["품목명"] == meta_item].copy()
                ac_T    = ac_item[ac_item["t_idx"] == 0]
                ac_feat_cols = [c for c in ac_T.columns if c.startswith("meta_ac_")]
                if len(ac_T) > 0:
                    for col in ac_feat_cols:
                        last_rows[col] = ac_T[col].iloc[0]

        # -----------------------------------------------
        # 예측
        # -----------------------------------------------
        for h in [1, 2, 3]:
            model    = all_models[item]["models"][h]
            use_cols = all_results[item][h]["use_cols"]

            # 실제 존재하는 feature만 사용
            available = [c for c in use_cols if c in last_rows.columns]
            X_test    = last_rows[available].fillna(0)

            pred_values = model.predict(X_test)
            final_pred  = max(float(np.mean(pred_values)), 0)
            horizon_preds[h][item] = final_pred

    for h in [1, 2, 3]:
        row = {"시점": f"{base_name}+{h}순"}
        for item in TARGET_MAP.keys():
            row[item] = horizon_preds[h][item]
        submission_rows.append(row)


############################################################
# 12. 제출 파일 저장
############################################################
submission = pd.DataFrame(submission_rows)
submission = submission[sample_sub.columns]

print("\n[submission 미리보기]")
print(submission.head(10).to_string(index=False))

submission.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")

[TRAIN 메타데이터 로드 중...]
  전국도매 집계 완료: (1404, 13)
  산지공판장 집계 완료: (1321, 13)

train_raw shape: (29376, 7)
sample_sub shape: (75, 11)

[건고추] 학습 시작
  데이터 shape: (144, 35)
  메타 feature 수: 0
  h=1 | train=140 | valid=3 | MAE=5120.5417 | RMSE=5950.7755 | features=25
  h=2 | train=139 | valid=3 | MAE=11574.8125 | RMSE=14698.0426 | features=25
  h=3 | train=138 | valid=3 | MAE=22350.0000 | RMSE=23694.4522 | features=25

[사과] 학습 시작
  데이터 shape: (144, 55)
  메타 feature 수: 20
  h=1 | train=139 | valid=3 | MAE=786.7910 | RMSE=1022.1214 | features=45
  h=2 | train=137 | valid=3 | MAE=1769.5645 | RMSE=1802.1743 | features=45
  h=3 | train=135 | valid=3 | MAE=923.3451 | RMSE=1030.0355 | features=45

[감자] 학습 시작
  데이터 shape: (144, 55)
  메타 feature 수: 20
  h=1 | train=140 | valid=3 | MAE=4646.9257 | RMSE=4904.8748 | features=45
  h=2 | train=139 | valid=3 | MAE=3003.8280 | RMSE=3808.5509 | features=45
  h=3 | train=138 | valid=3 | MAE=6024.0147 | RMSE=7302.1066 | features=45

[배] 학습 시작
  데이터 shape: (144, 55